# Freesound Audio Tagging 2019 — Predict & Submit

In [ ]:
# ================== CONFIG ==================
MODEL_PATH = "/kaggle/input/YOUR_MODEL_DATASET/fat2019_cnn.pt"
OUTPUT_NAME = "submission.csv"
BATCH_SIZE = 64
NUM_WORKERS = 2
TTA_CROPS = 3          # Test-time augmentation crops
CLIP_SECONDS = 5.0
N_MELS = 128
SAMPLE_RATE = 22050
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
import math, random, json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import resample_poly, stft
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
# ================== Audio & Feature Utils ==================

def hz_to_mel(freq):
    return 2595.0 * np.log10(1.0 + np.asarray(freq) / 700.0)

def mel_to_hz(mel):
    return 700.0 * (10.0 ** (np.asarray(mel) / 2595.0) - 1.0)

def mel_filterbank(sr, n_fft, n_mels, fmin, fmax):
    mel_points = np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    bins = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    bins = np.clip(bins, 0, n_fft // 2)
    fb = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)
    for i in range(1, n_mels + 1):
        left, center, right = bins[i - 1], bins[i], bins[i + 1]
        if center <= left: center = left + 1
        if right <= center: right = center + 1
        right = min(right, n_fft // 2)
        if center > left:
            fb[i - 1, left:center] = (np.arange(left, center) - left) / (center - left)
        if right > center:
            fb[i - 1, center:right] = (right - np.arange(center, right)) / (right - center)
    return fb

def read_wav_mono(path, target_sr):
    sr, audio = wavfile.read(path)
    if audio.ndim > 1: audio = audio.mean(axis=1)
    audio = audio.astype(np.float32)
    if audio.size == 0: return np.zeros(target_sr, dtype=np.float32)
    peak = np.max(np.abs(audio))
    if peak > 0: audio = audio / peak
    if sr != target_sr:
        gcd = math.gcd(sr, target_sr)
        audio = resample_poly(audio, target_sr // gcd, sr // gcd).astype(np.float32)
    return audio

class LogMelExtractor:
    def __init__(self, sample_rate=22050, n_fft=1024, hop_length=512, n_mels=128):
        self.sample_rate = sample_rate
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.fb = mel_filterbank(sample_rate, n_fft, n_mels, 20.0, sample_rate / 2.0)

    def __call__(self, audio):
        if audio.size < self.n_fft:
            audio = np.pad(audio, (0, self.n_fft - audio.size))
        _, _, zxx = stft(audio, fs=self.sample_rate, nperseg=self.n_fft,
                          noverlap=self.n_fft - self.hop_length, nfft=self.n_fft,
                          boundary=None, padded=False)
        power = np.abs(zxx).astype(np.float32) ** 2
        mel = np.maximum(self.fb @ power, 1e-10)
        logmel = np.log(mel)
        mean, std = logmel.mean(), logmel.std() + 1e-6
        return ((logmel - mean) / std).astype(np.float32)

In [ ]:
# ================== Dataset ==================

class TestDataset(Dataset):
    """Lightweight dataset for test-time prediction only."""
    def __init__(self, rows, clip_seconds, sample_rate=22050, n_mels=128, tta_crops=1):
        self.rows = rows.reset_index(drop=True)
        self.clip_samples = int(sample_rate * clip_seconds)
        self.sample_rate = sample_rate
        self.tta_crops = tta_crops
        self.extractor = LogMelExtractor(sample_rate=sample_rate, n_mels=n_mels)

    def __len__(self):
        return len(self.rows)

    def _crop_audio(self, audio, crop_index=0):
        if audio.size < self.clip_samples:
            return np.pad(audio, (0, self.clip_samples - audio.size))
        if audio.size == self.clip_samples:
            return audio
        max_start = audio.size - self.clip_samples
        if self.tta_crops <= 1:
            start = max_start // 2
        else:
            start = round(max_start * crop_index / (self.tta_crops - 1))
        return audio[start : start + self.clip_samples]

    def _load_item(self, index, crop_index=0):
        row = self.rows.iloc[index]
        audio = read_wav_mono(Path(row.audio_dir) / row.fname, self.sample_rate)
        audio = self._crop_audio(audio, crop_index)
        logmel = self.extractor(audio)
        return torch.from_numpy(logmel).unsqueeze(0)

    def __getitem__(self, index):
        if self.tta_crops > 1:
            crops = [self._load_item(index, i) for i in range(self.tta_crops)]
            x = torch.stack(crops, dim=0)
        else:
            x = self._load_item(index)
        return x, self.rows.iloc[index].fname

In [ ]:
# ================== Model ==================

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        ]
        if pool: layers.append(nn.MaxPool2d(2))
        layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class ResBlock(nn.Module):
    def __init__(self, channels, dropout, pool=False):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()
        self.dropout = nn.Dropout2d(dropout)
    def forward(self, x):
        out = self.conv(x) + x
        out = self.act(out)
        return self.dropout(self.pool(out))

class DeepAudioCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.25):
        super().__init__()
        d = dropout
        self.block1 = ConvBlock(1, 64, d * 0.5, pool=True)
        self.block2 = ConvBlock(64, 128, d * 0.5, pool=True)
        self.block3 = ConvBlock(128, 256, d, pool=True)
        self.block4 = ConvBlock(256, 512, d, pool=True)
        self.resblocks = nn.Sequential(
            ResBlock(512, d), ResBlock(512, d), ResBlock(512, d),
        )
        self.block5 = ConvBlock(512, 512, d, pool=False)
        self.block6 = ConvBlock(512, 768, d, pool=False)
        self.head = nn.Sequential(
            nn.Linear(1536, 512), nn.ReLU(inplace=True), nn.Dropout(d),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(d * 0.5),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = self.block4(x); x = self.resblocks(x)
        x = self.block5(x); x = self.block6(x)
        avg = torch.mean(x, dim=(2, 3))
        maxv = torch.amax(x, dim=(2, 3))
        return self.head(torch.cat([avg, maxv], dim=1))

print("Model defined.")

In [ ]:
# ================== Load Model ==================

device = torch.device(DEVICE)
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
classes = checkpoint["classes"]
train_args = checkpoint.get("args", {})

# Respect saved training config if available, else use notebook defaults
n_mels = int(train_args.get("n_mels", N_MELS))
clip_seconds = float(train_args.get("clip_seconds", CLIP_SECONDS))
dropout = float(train_args.get("dropout", 0.25))

model = DeepAudioCNN(len(classes), dropout=dropout).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

print(f"Loaded model: {len(classes)} classes")
print(f"Best LWLRAP (from training): {checkpoint.get('best_lwlrap', 'N/A')}")

In [ ]:
# ================== Prepare Test Data ==================
# Kaggle 比赛数据默认路径
BASE = Path("/kaggle/input/freesound-audio-tagging-2019")

test_df = pd.read_csv(BASE / "sample_submission.csv")
test_df = test_df[["fname"]].copy()
test_df["audio_dir"] = str(BASE / "test")

# Filter to existing files only
exists = test_df["fname"].map(lambda n: (Path(test_df["audio_dir"].iloc[0]) / n).exists())
test_df = test_df.loc[exists].reset_index(drop=True)

print(f"Test files found: {len(test_df)}")

In [ ]:
# ================== Predict ==================

ds = TestDataset(test_df, clip_seconds, n_mels=n_mels, tta_crops=TTA_CROPS)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, collate_fn=lambda b: (torch.stack([x[0] for x in b]), [x[1] for x in b]))

all_fnames, all_scores = [], []
with torch.no_grad():
    for x, fnames in tqdm(loader, desc="predict"):
        if x.ndim == 5:  # TTA: (batch, crops, 1, mel, time)
            B, C = x.shape[0], x.shape[1]
            x = x.view(B * C, 1, x.shape[-2], x.shape[-1]).to(device)
            scores = torch.sigmoid(model(x)).view(B, C, -1).mean(dim=1)
        else:
            scores = torch.sigmoid(model(x.to(device)))
        all_fnames.extend(fnames)
        all_scores.append(scores.cpu().numpy())

sub = pd.DataFrame(np.concatenate(all_scores, axis=0), columns=classes)
sub.insert(0, "fname", all_fnames)

print(f"\nSubmissions: {len(sub)} rows × {len(classes)} labels")

In [ ]:
# ================== Export ==================

sub.to_csv(OUTPUT_NAME, index=False)
print(f"Saved: {OUTPUT_NAME}")

# Quick sanity check
print(f"\nSample (first 3 rows × first 5 labels):")
display(sub.iloc[:3, :6])